# mt103-extractor-lora: QLoRA fine-tuning on Kaggle (free GPU)

**What this does:** fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA (4-bit, rank 16, alpha 32, dropout 0.05,
lr 2e-4 cosine, max length 1024, 3 epochs) to turn messy payment text into MT103 JSON. Same code and config as
`make train` in the repo; this notebook is generated from the repo by `notebooks/build_notebook.py`.

**How to run**
1. kaggle.com, then **Create, New Notebook**, then **File, Import Notebook**, and upload this `.ipynb`.
2. Right sidebar: **Session options, Accelerator, GPU T4 x2**. Turn **Internet on** (needs phone verification).
3. **Run All**. It takes about 1 to 1.5 hours on a T4.
4. When it finishes, open the **Output** panel (`/kaggle/working`) and download **`adapter.zip`** (and
   `train_log.json`, `loss.png` if you want the curves).
5. In the repo: unzip into `models/adapter/` (so `models/adapter/adapter_config.json` exists), then run
   `make merge serve eval-ft` (see TESTING.md).

**Data:** the dataset is SYNTHETIC. It is regenerated here with the repo's seeded generator (seed 103), and the
SHA-256 of each split is checked against the repo's copy, so you don't need to upload anything. If you'd rather
upload `data/*.jsonl` as a Kaggle dataset, set `DATA_DIR` in the data cell to its `/kaggle/input/...` path.

In [ ]:
!pip install -q "transformers==5.24.0" "peft==0.22.0" "trl==1.18.2" "datasets" "accelerate" "bitsandbytes" "pydantic>=2" pyyaml matplotlib
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4 is enough; avoids naive model-parallel across 2 GPUs
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working

## Project source (written from the repo)

In [ ]:
import os
for d in ['mt103/data', 'mt103/evals', 'mt103/train', 'mt103/models', 'mt103/results']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile mt103/schema.py
"""Pydantic schema for the subset of SWIFT MT103 fields used by this project.

Fields (tag -> key):
  20  sender_reference        23B bank_operation_code
  32A value_date / currency / amount
  50K ordering_customer       52A ordering_institution (BIC)
  57A account_with_institution (BIC)
  59  beneficiary             70  remittance_info
  71A charges (OUR/SHA/BEN)

An extraction result is {"fields": MT103Fields-like dict, "errors": [str, ...]}.
Fields that are missing or impossible in the source text are null, and impossible
values are reported in `errors` (never invented).
"""
from __future__ import annotations

import datetime as dt
import re
from typing import Literal, Optional

from pydantic import BaseModel, ConfigDict, Field, field_validator

# ISO 4217 active alphabetic codes (complete list as of 2024, minus funds/precious metals).
ISO4217 = frozenset("""
AED AFN ALL AMD ANG AOA ARS AUD AWG AZN BAM BBD BDT BGN BHD BIF BMD BND BOB BRL BSD BTN BWP BYN
BZD CAD CDF CHF CLP CNY COP CRC CUP CVE CZK DJF DKK DOP DZD EGP ERN ETB EUR FJD FKP GBP GEL GHS
GIP GMD GNF GTQ GYD HKD HNL HTG HUF IDR ILS INR IQD IRR ISK JMD JOD JPY KES KGS KHR KMF KPW KRW
KWD KYD KZT LAK LBP LKR LRD LSL LYD MAD MDL MGA MKD MMK MNT MOP MRU MUR MVR MWK MXN MYR MZN NAD
NGN NIO NOK NPR NZD OMR PAB PEN PGK PHP PKR PLN PYG QAR RON RSD RUB RWF SAR SBD SCR SDG SEK SGD
SHP SLE SOS SRD SSP STN SVC SYP SZL THB TJS TMT TND TOP TRY TTD TWD TZS UAH UGX USD UYU UZS VES
VND VUV WST XAF XCD XOF XPF YER ZAR ZMW ZWL
""".split())

BIC_RE = re.compile(r"^[A-Z]{4}[A-Z]{2}[A-Z0-9]{2}([A-Z0-9]{3})?$")
REF_RE = re.compile(r"^[A-Za-z0-9/\-?:().,'+ ]{1,16}$")
BANK_OP_CODES = ("CRED", "CRTS", "SPAY", "SPRI", "SSTD")
FIELD_KEYS = (
    "sender_reference", "bank_operation_code", "value_date", "currency", "amount",
    "ordering_customer", "ordering_institution", "account_with_institution",
    "beneficiary", "remittance_info", "charges",
)
REQUIRED_KEYS = ("sender_reference", "value_date", "currency", "amount",
                 "ordering_customer", "beneficiary", "charges")
# Machine-readable error codes. Each `errors` entry is "<CODE>: <field>: <detail>", e.g.
# "INVALID_CURRENCY: currency: USX is not ISO-4217". Scoring uses (CODE, field) only.
ERROR_CODES = ("INVALID_CURRENCY", "INVALID_DATE", "INVALID_BIC", "NON_POSITIVE_AMOUNT",
               "INVALID_CHARGES", "MISSING_REQUIRED")


class Party(BaseModel):
    model_config = ConfigDict(extra="forbid")
    account: Optional[str] = None
    name: str = Field(min_length=1, max_length=140)
    address: Optional[str] = None


class MT103Fields(BaseModel):
    """All fields optional: an extractor may legitimately leave a field null."""
    model_config = ConfigDict(extra="forbid")

    sender_reference: Optional[str] = Field(None, description=":20:")
    bank_operation_code: Optional[Literal["CRED", "CRTS", "SPAY", "SPRI", "SSTD"]] = Field(None, description=":23B:")
    value_date: Optional[str] = Field(None, description=":32A: date, ISO YYYY-MM-DD")
    currency: Optional[str] = Field(None, description=":32A: ISO-4217")
    amount: Optional[float] = Field(None, description=":32A: amount, > 0")
    ordering_customer: Optional[Party] = Field(None, description=":50K:")
    ordering_institution: Optional[str] = Field(None, description=":52A: BIC")
    account_with_institution: Optional[str] = Field(None, description=":57A: BIC")
    beneficiary: Optional[Party] = Field(None, description=":59:")
    remittance_info: Optional[str] = Field(None, max_length=140, description=":70:")
    charges: Optional[Literal["OUR", "SHA", "BEN"]] = Field(None, description=":71A:")

    @field_validator("sender_reference")
    @classmethod
    def _ref(cls, v):
        if v is not None and not REF_RE.match(v):
            raise ValueError("sender_reference must be 1-16 SWIFT x-chars")
        return v

    @field_validator("value_date")
    @classmethod
    def _date(cls, v):
        if v is not None:
            dt.date.fromisoformat(v)  # raises on bad format / impossible date
            if not re.match(r"^\d{4}-\d{2}-\d{2}$", v):
                raise ValueError("value_date must be YYYY-MM-DD")
        return v

    @field_validator("currency")
    @classmethod
    def _ccy(cls, v):
        if v is not None and v not in ISO4217:
            raise ValueError(f"{v} is not an ISO-4217 currency")
        return v

    @field_validator("amount")
    @classmethod
    def _amt(cls, v):
        if v is not None and not v > 0:
            raise ValueError("amount must be positive")
        return v

    @field_validator("ordering_institution", "account_with_institution")
    @classmethod
    def _bic(cls, v):
        if v is not None and not BIC_RE.match(v):
            raise ValueError(f"{v} is not a valid BIC")
        return v


class Extraction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    fields: MT103Fields
    errors: list[str] = Field(default_factory=list)

    @field_validator("errors")
    @classmethod
    def _codes(cls, v):
        for e in v:
            if e.split(":", 1)[0].strip() not in ERROR_CODES:
                raise ValueError(f"unknown error code in {e!r}")
        return v


def error_keys(errors) -> set[str]:
    """Normalise an errors list to a set of "CODE:field" keys (detail text ignored)."""
    out = set()
    if not isinstance(errors, list):
        return out
    for e in errors:
        if not isinstance(e, str):
            continue
        parts = [x.strip() for x in e.split(":")]
        code = parts[0].upper().replace(" ", "_")
        field = parts[1].split()[0].lower() if len(parts) > 1 and parts[1] else ""
        out.add(f"{code}:{field}")
    return out

In [ ]:
%%writefile mt103/prompts.py
"""Prompt construction and output parsing shared by training, eval and serving."""
from __future__ import annotations

import json
import re

SYSTEM = """You convert messy payment instructions (emails, chats, partial SWIFT text, tickets) into SWIFT MT103 JSON.
Return ONLY one JSON object, no prose, with exactly this shape:
{"fields":{"sender_reference":str|null,"bank_operation_code":"CRED"|"CRTS"|"SPAY"|"SPRI"|"SSTD"|null,"value_date":"YYYY-MM-DD"|null,"currency":ISO-4217 str|null,"amount":number|null,"ordering_customer":{"account":str|null,"name":str,"address":str|null}|null,"ordering_institution":BIC|null,"account_with_institution":BIC|null,"beneficiary":{"account":str|null,"name":str,"address":str|null}|null,"remittance_info":str|null,"charges":"OUR"|"SHA"|"BEN"|null},"errors":[str]}
Rules:
- 20=sender_reference, 23B=bank_operation_code, 32A=value_date+currency+amount, 50K=ordering_customer, 52A=ordering_institution (sender's bank BIC), 57A=account_with_institution (beneficiary's bank BIC), 59=beneficiary, 70=remittance_info, 71A=charges.
- Numeric dates are day-first (dd/mm/yyyy). SWIFT 6-digit dates are YYMMDD. Output dates as YYYY-MM-DD.
- Amounts: plain number. Indian grouping 1,25,000.50 = 125000.5; "2.5 lakh" = 250000; European 1.234,56 = 1234.56. Symbols: $=USD, €=EUR, £=GBP, Rs/₹=INR, ¥=JPY.
- Charges: ours/we pay=OUR, shared=SHA, beneficiary pays=BEN.
- Copy names, accounts, addresses and references exactly as written. A field not in the text is null. Never invent values.
- If a value is impossible, set the field to null and add an error "CODE: field: detail". Codes: INVALID_CURRENCY (not ISO-4217), INVALID_DATE (not a real date), INVALID_BIC (not 8 or 11 chars AAAABBCC[XXX]), NON_POSITIVE_AMOUNT (zero or negative), INVALID_CHARGES (not OUR/SHA/BEN), MISSING_REQUIRED (value_date, beneficiary or charges not stated).
- errors is [] when everything is fine."""

# Fixed few-shot ids from the TRAIN split (email, chat with Indian amount format, partial SWIFT, ticket with an error).
FEWSHOT_IDS = ("syn-00004", "syn-00012", "syn-00000", "syn-00005")


def target_str(target: dict) -> str:
    """Canonical compact JSON for a target (what the fine-tuned model is trained to emit)."""
    return json.dumps(target, ensure_ascii=False, separators=(",", ":"))


def chat_messages(text: str, fewshot: list[dict] | None = None) -> list[dict]:
    msgs = [{"role": "system", "content": SYSTEM}]
    for ex in fewshot or []:
        msgs.append({"role": "user", "content": ex["text"]})
        msgs.append({"role": "assistant", "content": target_str(ex["target"])})
    msgs.append({"role": "user", "content": text})
    return msgs


_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.S)


def parse_output(raw: str):
    """Return (obj or None). Accepts fenced code blocks and leading/trailing prose."""
    if raw is None:
        return None
    s = raw.strip()
    m = _FENCE.search(s)
    if m:
        s = m.group(1).strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    start = s.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(s)):
        c = s[i]
        if in_str:
            if esc:
                esc = False
            elif c == "\\":
                esc = True
            elif c == '"':
                in_str = False
            continue
        if c == '"':
            in_str = True
        elif c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(s[start:i + 1])
                except Exception:
                    return None
    return None


# Short system prompt used for the FINE-TUNED model (training, Ollama Modelfile, eval with
# --prompt ft). The format and error rules are learned from data, so the prompt stays tiny:
# that is what makes the fine-tuned model much cheaper per request than few-shot prompting.
FT_SYSTEM = "Extract the payment as MT103 JSON {fields, errors}. Null for absent fields; flag impossible values in errors."


def ft_messages(text: str) -> list[dict]:
    return [{"role": "system", "content": FT_SYSTEM}, {"role": "user", "content": text}]

In [ ]:
%%writefile mt103/data/__init__.py

In [ ]:
%%writefile mt103/data/generate.py
"""Seeded generator of SYNTHETIC "messy payment text -> MT103 JSON" examples.

Everything produced here is synthetic. Names, accounts, references and amounts are
random; BICs are reused from the swift-mt103-monitor simulator's bank list plus a few
other well-known public BICs. No real payment data is used.

Usage:
    python data/generate.py --n 3000 --seed 103 --out data/
Writes data/{train,val,test}.jsonl (80/10/10) and data/MANIFEST.json.
"""
from __future__ import annotations

import argparse
import datetime as dt
import hashlib
import json
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from schema import ISO4217, Extraction  # noqa: E402

# Bank list from swift-mt103-monitor/swift/producer.go, extended.
BANKS_SRC = ["DEUTDEDB", "CHASUS33", "BARCGB22", "BNPAFRPP", "HSBCHKHH"]
BANKS_EXTRA = ["SBININBB", "HDFCINBB", "ICICINBBXXX", "UTIBINBB", "CITIUS33", "DEUTDEFF",
               "BOFAUS3N", "SCBLSGSG", "MHCBJPJT", "UBSWCHZH80A", "INGBNL2A", "COBADEFFXXX"]
BANKS = BANKS_SRC + BANKS_EXTRA
BAD_BICS = ["HDFC123", "CHASUS", "BARC-GB22", "1234INBB", "DEUTDEDBX", "SBIN IN BB", "HSBCHK"]

# currency -> (weight, typical amount range)
CCY = {"USD": (40, (10, 10000)), "EUR": (14, (50, 250000)), "GBP": (8, (50, 150000)),
       "INR": (20, (1000, 9000000)), "SGD": (4, (100, 200000)), "JPY": (4, (10000, 9000000)),
       "CHF": (3, (100, 100000)), "AED": (4, (500, 400000)), "HKD": (3, (500, 500000))}
CCY_SYMBOL = {"USD": ["$", "US$"], "EUR": ["€"], "GBP": ["£"], "INR": ["Rs.", "₹", "INR", "Rs"],
              "JPY": ["¥"]}
BAD_CCY = ["USX", "EUD", "RSS", "GBR", "IND", "DOL", "UDS", "ERU"]

FIRST = ["Aarav", "Priya", "Rohan", "Ananya", "Vikram", "Meera", "John", "Sarah", "Lukas", "Chloe",
         "Kenji", "Fatima", "Omar", "Elena", "Arjun", "Divya", "Marco", "Aisha", "Wei", "Nikhil"]
LAST = ["Sharma", "Iyer", "Patel", "Reddy", "Kapoor", "Smith", "Müller", "Dubois", "Tanaka",
        "Khan", "Rossi", "Nair", "Chen", "Gupta", "Fernandes", "Menon", "Brown", "Schmidt"]
CO_A = ["Arcadia", "Bluepeak", "Coromandel", "Deccan", "Everest", "Falcon", "Ganges", "Helix",
        "Indus", "Juniper", "Kestrel", "Lotus", "Meridian", "Nilgiri", "Orion", "Saffron",
        "Tata-Vale", "Zenith", "Northwind", "Harbor"]
CO_B = ["Textiles", "Logistics", "Pharma", "Steel", "Software", "Exports", "Agro", "Capital",
        "Trading", "Components", "Foods", "Shipping", "Analytics", "Motors"]
CO_C = ["Pvt Ltd", "Ltd", "GmbH", "LLC", "Inc", "SA", "Pte Ltd", "LLP", "BV", "PLC"]
CITY = ["Mumbai 400001 IN", "Bengaluru 560001 IN", "Pune 411001 IN", "Chennai 600002 IN",
        "Frankfurt 60311 DE", "London EC2V 7HN GB", "New York NY 10005 US", "Paris 75009 FR",
        "Singapore 048583 SG", "Hong Kong HK", "Dubai AE", "Zurich 8001 CH", "Tokyo 100-0005 JP"]
STREET = ["12 MG Road", "4th Floor, Nariman Point", "221 Park Street", "Plot 7, MIDC",
          "Taunusanlage 12", "1 Churchill Place", "270 Park Avenue", "16 Bd des Italiens",
          "8 Marina Blvd", "Unit 5B, Hi-Tech Park"]
REMIT = ["INV {n} consulting fees", "Invoice {n} / spare parts", "PO {n} advance payment",
         "Salary {mon} 2026", "Rent {mon} 2026 office", "Refund order {n}",
         "Freight charges BL {n}", "Licence fee Q{q} 2026", "Settlement of INV {n} and INV {n2}",
         "Tuition fees semester {q}", "Dividend payout {mon}", "Software subscription {n}"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MONTHS_LONG = ["January", "February", "March", "April", "May", "June", "July", "August",
               "September", "October", "November", "December"]
CHARGE_WORDS = {"OUR": ["OUR", "our", "ours", "OUR (we pay all charges)", "all charges on us"],
                "SHA": ["SHA", "sha", "shared", "SHA - shared", "split charges (SHA)"],
                "BEN": ["BEN", "ben", "beneficiary pays", "BEN (charges to beneficiary)"]}
BAD_CHARGES = ["FULL", "NONE", "OURS+BEN", "XYZ", "HALF"]
TYPOS = {"beneficiary": ["benificiary", "beneficary", "bene", "benef."], "amount": ["amt", "amout", "amnt"],
         "please": ["pls", "plz", "please"], "account": ["acct", "a/c", "acc"],
         "reference": ["ref", "refrence", "ref no"], "payment": ["pymt", "paymnt", "payment"],
         "charges": ["charges", "chrgs", "charge"], "value date": ["value date", "val dt", "VD", "value dt"]}


def indian_format(x: float, decimals: bool = True) -> str:
    s = f"{x:.2f}"
    whole, frac = s.split(".")
    if len(whole) > 3:
        head, tail = whole[:-3], whole[-3:]
        groups = []
        while len(head) > 2:
            groups.insert(0, head[-2:])
            head = head[:-2]
        if head:
            groups.insert(0, head)
        whole = ",".join(groups) + "," + tail
    return f"{whole}.{frac}" if decimals or frac != "00" else whole


class Gen:
    def __init__(self, seed: int):
        self.r = random.Random(seed)

    # ---------- primitives ----------
    def w(self, key: str) -> str:
        return self.r.choice(TYPOS[key]) if self.r.random() < 0.5 else key

    def company(self) -> str:
        r = self.r
        return f"{r.choice(CO_A)} {r.choice(CO_B)} {r.choice(CO_C)}"

    def person(self) -> str:
        return f"{self.r.choice(FIRST)} {self.r.choice(LAST)}"

    def account(self) -> str:
        r = self.r
        k = r.random()
        if k < 0.4:
            return "".join(r.choice("0123456789") for _ in range(r.choice([11, 12, 14, 15])))
        if k < 0.75:
            cc = r.choice(["DE", "GB", "FR", "NL", "CH"])
            return cc + "".join(r.choice("0123456789") for _ in range(r.choice([18, 20])))
        return "".join(r.choice("0123456789") for _ in range(9)) + "-" + str(r.randint(10, 99))

    def party(self, need_account: bool = True) -> dict:
        r = self.r
        name = self.company() if r.random() < 0.7 else self.person()
        acct = self.account() if (need_account or r.random() < 0.7) else None
        addr = f"{r.choice(STREET)}, {r.choice(CITY)}" if r.random() < 0.55 else None
        return {"account": acct, "name": name, "address": addr}

    def ref(self) -> str:
        r = self.r
        k = r.random()
        if k < 0.4:
            return f"TXN{r.randint(0, 999999):06d}"  # same pattern as swift-mt103-monitor
        if k < 0.7:
            return f"FT26{r.randint(10000000, 99999999)}"[:16]
        if k < 0.9:
            return f"PAY-{r.randint(2025, 2026)}-{r.randint(1, 99999):05d}"
        return f"OPS/{r.choice(MONTHS).upper()}/{r.randint(1, 999)}"

    def amount(self, ccy: str) -> float:
        lo, hi = CCY[ccy][1]
        x = round(self.r.uniform(lo, hi), 2)
        if ccy == "JPY":
            x = float(int(x))
        if self.r.random() < 0.25:
            x = float(round(x, -2)) or 100.0
        return x

    def fmt_amount(self, x: float, ccy: str) -> str:
        r = self.r
        k = r.random()
        if ccy == "INR" and k < 0.55:
            if x >= 100000 and r.random() < 0.2 and round(x / 100000, 2) * 100000 == x:
                return f"{x / 100000:g} lakh"
            return indian_format(x, decimals=r.random() < 0.7)
        if ccy == "EUR" and k < 0.25:
            s = f"{x:,.2f}"
            return s.replace(",", "_").replace(".", ",").replace("_", ".")
        if k < 0.6:
            return f"{x:,.2f}"
        if k < 0.85:
            return f"{x:.2f}" if r.random() < 0.6 else f"{x:g}"
        return f"{x:,.2f}".replace(",", " ")

    def fmt_ccy(self, ccy: str) -> str:
        r = self.r
        if ccy in CCY_SYMBOL and r.random() < 0.35:
            return r.choice(CCY_SYMBOL[ccy])
        return ccy if r.random() < 0.8 else ccy.lower()

    def date(self) -> dt.date:
        return dt.date(2026, 1, 1) + dt.timedelta(days=self.r.randint(0, 420))

    def fmt_date(self, d: dt.date) -> str:
        r = self.r
        k = r.randint(0, 7)
        return [d.isoformat(), d.strftime("%d/%m/%Y"), d.strftime("%d-%m-%Y"), d.strftime("%d.%m.%Y"),
                f"{d.day} {MONTHS[d.month - 1]} {d.year}", f"{MONTHS_LONG[d.month - 1]} {d.day}, {d.year}",
                f"{d.day:02d}-{MONTHS[d.month - 1]}-{d.strftime('%y')}", d.strftime("%y%m%d")][k]

    def bad_date(self) -> str:
        r = self.r
        y = r.choice([2026, 2027])
        return r.choice([f"31/04/{y}", f"30/02/{y}", f"{y}-13-05", f"32.01.{y}", f"31 Jun {y}",
                         f"29/02/{y}" if y == 2026 else f"31/09/{y}", f"{y}-00-12", f"31-11-{y}"])

    def remit(self) -> str:
        r = self.r
        return r.choice(REMIT).format(n=f"{r.randint(1, 9999):04d}", n2=f"{r.randint(1, 9999):04d}",
                                      mon=r.choice(MONTHS), q=r.randint(1, 4))

    # ---------- one example ----------
    def example(self, idx: int) -> dict:
        r = self.r
        ccy = r.choices(list(CCY), weights=[v[0] for v in CCY.values()])[0]
        amt = self.amount(ccy)
        d = self.date()
        f = {
            "sender_reference": self.ref(),
            "bank_operation_code": "CRED" if r.random() < 0.9 else r.choice(["SPAY", "SPRI", "SSTD", "CRTS"]),
            "value_date": d.isoformat(), "currency": ccy, "amount": amt,
            "ordering_customer": self.party(), "ordering_institution": r.choice(BANKS),
            "account_with_institution": r.choice(BANKS), "beneficiary": self.party(),
            "remittance_info": self.remit(), "charges": r.choice(["OUR", "SHA", "SHA", "BEN"]),
        }
        # surface strings (what appears in the text)
        surf = {"date": self.fmt_date(d), "ccy": self.fmt_ccy(ccy), "amt": self.fmt_amount(amt, ccy),
                "charges": r.choice(CHARGE_WORDS[f["charges"]])}
        # optional fields absent from text -> null in target, no error
        for k, p in (("bank_operation_code", 0.45), ("ordering_institution", 0.3),
                     ("account_with_institution", 0.3), ("remittance_info", 0.2)):
            if r.random() < p:
                f[k] = None
        errors: list[str] = []
        # impossible values (~22% of examples have at least one)
        if r.random() < 0.22:
            kind = r.choice(["ccy", "date", "bic", "amt", "charges", "missing"])
            if kind == "ccy":
                bad = r.choice(BAD_CCY); assert bad not in ISO4217
                surf["ccy"] = bad; f["currency"] = None
                errors.append(f"INVALID_CURRENCY: currency: {bad} is not an ISO-4217 code")
            elif kind == "date":
                surf["date"] = self.bad_date(); f["value_date"] = None
                errors.append(f"INVALID_DATE: value_date: {surf['date']} is not a real calendar date")
            elif kind == "bic":
                which = r.choice(["ordering_institution", "account_with_institution"])
                bad = r.choice(BAD_BICS); f[which] = None
                surf["badbic"] = (which, bad)
                errors.append(f"INVALID_BIC: {which}: {bad} is not a valid BIC")
            elif kind == "amt":
                bad = r.choice(["0.00", "-" + self.fmt_amount(amt, ccy), "0", "-" + f"{amt:.2f}"])
                surf["amt"] = bad; f["amount"] = None
                errors.append(f"NON_POSITIVE_AMOUNT: amount: {bad} is not a positive amount")
            elif kind == "charges":
                bad = r.choice(BAD_CHARGES); surf["charges"] = bad; f["charges"] = None
                errors.append(f"INVALID_CHARGES: charges: {bad} is not OUR/SHA/BEN")
            else:
                miss = r.choice(["charges", "beneficiary", "value_date"])
                f[miss] = None; surf["missing"] = miss
                errors.append(f"MISSING_REQUIRED: {miss}: not stated in the text")
        text = self.render(f, surf)
        target = {"fields": f, "errors": errors}
        Extraction.model_validate(target)  # every target must be schema-valid
        return {"id": f"syn-{idx:05d}", "synthetic": True, "text": text, "target": target}

    # ---------- renderers ----------
    def bic_s(self, f, surf, key):
        if "badbic" in surf and surf["badbic"][0] == key:
            return surf["badbic"][1]
        return f[key]

    def party_s(self, p, sep=", "):
        bits = []
        if p.get("account"):
            bits.append(p["account"] if self.r.random() < 0.5 else f"{self.w('account')} {p['account']}")
        bits.append(p["name"])
        if p.get("address"):
            bits.append(p["address"])
        return sep.join(bits)

    def render(self, f, surf) -> str:
        style = self.r.choices(["email", "chat", "swift", "ticket", "forward"], weights=[30, 25, 20, 15, 10])[0]
        return getattr(self, f"r_{style}")(f, surf).strip()

    def money(self, surf):
        c, a = surf["ccy"], surf["amt"]
        if c in ("$", "US$", "€", "£", "₹", "¥", "Rs.", "Rs"):
            return f"{c}{a}" if self.r.random() < 0.6 else f"{c} {a}"
        return f"{c} {a}" if self.r.random() < 0.7 else f"{a} {c}"

    def r_email(self, f, s):
        r, w = self.r, self.w
        oc, bn = f["ordering_customer"], f["beneficiary"]
        lines = [r.choice(["Hi team,", "Dear Ops,", "Hello,", "Good morning,", "Hi"]), ""]
        lines.append(f"{w('please').capitalize()} process the below {w('payment')} today.")
        lines.append(f"Our {w('reference')}: {f['sender_reference']}")
        lines.append(f"{w('amount').capitalize()}: {self.money(s)}")
        if s.get("missing") != "value_date":
            lines.append(f"{w('value date').capitalize()}: {s['date']}")
        lines.append(f"Debit {w('account')} / remitter: {self.party_s(oc)}")
        if f["ordering_institution"] or "badbic" in s and s["badbic"][0] == "ordering_institution":
            lines.append(f"Our bank: {self.bic_s(f, s, 'ordering_institution')}")
        if s.get("missing") != "beneficiary":
            lines.append(f"{w('beneficiary').capitalize()}: {self.party_s(bn)}")
        if f["account_with_institution"] or "badbic" in s and s["badbic"][0] == "account_with_institution":
            lines.append(f"{w('beneficiary').capitalize()} bank SWIFT: {self.bic_s(f, s, 'account_with_institution')}")
        if f["remittance_info"]:
            lines.append(f"Purpose: {f['remittance_info']}")
        if s.get("missing") != "charges":
            lines.append(f"{w('charges').capitalize()}: {s['charges']}")
        if f["bank_operation_code"]:
            lines.append(f"Op code {f['bank_operation_code']}")
        body = lines[:2] + r.sample(lines[2:], len(lines) - 2) if r.random() < 0.3 else lines
        body += ["", r.choice(["Thanks,", "Regards,", "Thx", "Best,"]), self.person()]
        return "\n".join(body)

    def r_chat(self, f, s):
        r, w = self.r, self.w
        oc, bn = f["ordering_customer"], f["beneficiary"]
        parts = [r.choice(["hey", "urgent:", "pls rush -", "@ops", "quick one"]),
                 f"send {self.money(s)} to {self.party_s(bn, ' ') if s.get('missing') != 'beneficiary' else 'the usual guys'}"]
        if f["account_with_institution"] or "badbic" in s and s["badbic"][0] == "account_with_institution":
            parts.append(f"at {self.bic_s(f, s, 'account_with_institution')}")
        parts.append(f"from {self.party_s(oc, ' ')}")
        if f["ordering_institution"] or "badbic" in s and s["badbic"][0] == "ordering_institution":
            parts.append(f"via {self.bic_s(f, s, 'ordering_institution')}")
        if s.get("missing") != "value_date":
            parts.append(f"{w('value date')} {s['date']}")
        parts.append(f"{w('reference')} {f['sender_reference']}")
        if f["remittance_info"]:
            parts.append(f"for {f['remittance_info']}")
        if s.get("missing") != "charges":
            parts.append(f"{w('charges')} {s['charges']}")
        if f["bank_operation_code"]:
            parts.append(f"({f['bank_operation_code']})")
        txt = " ".join(parts)
        return txt.lower() if r.random() < 0.3 else txt

    def r_swift(self, f, s):
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        ccy = s["ccy"] if len(s["ccy"]) == 3 and s["ccy"].isalpha() else f["currency"] or s["ccy"]
        ccy = ccy.upper()
        amt = s["amt"]
        if not amt.startswith("-") and "lakh" not in amt:
            amt = f"{f['amount']:.2f}".replace(".", ",") if f["amount"] else amt
        date = s["date"]
        if f["value_date"] and r.random() < 0.7:
            date = dt.date.fromisoformat(f["value_date"]).strftime("%y%m%d")
        out = []
        if r.random() < 0.4 and f["ordering_institution"] and f["account_with_institution"]:
            # header only when both BICs are real, so it never leaks a BIC the target says is absent
            out.append(f"{{1:F01{f['ordering_institution'][:8]}AXXX0000000000}}{{2:I103{f['account_with_institution'][:8]}XN}}{{4:")
        out.append(f":20:{f['sender_reference']}")
        if f["bank_operation_code"]:
            out.append(f":23B:{f['bank_operation_code']}")
        if s.get("missing") != "value_date":
            out.append(f":32A:{date}{ccy}{amt}")
        else:
            out.append(f":32A:{ccy}{amt}  (date tbc)")
        out.append(":50K:" + "\n".join(x for x in [("/" + oc["account"]) if oc["account"] else None, oc["name"], oc["address"]] if x))
        if f["ordering_institution"] or "badbic" in s and s["badbic"][0] == "ordering_institution":
            out.append(f":52A:{self.bic_s(f, s, 'ordering_institution')}")
        if f["account_with_institution"] or "badbic" in s and s["badbic"][0] == "account_with_institution":
            out.append(f":57A:{self.bic_s(f, s, 'account_with_institution')}")
        if s.get("missing") != "beneficiary":
            out.append(":59:" + "\n".join(x for x in [("/" + bn["account"]) if bn["account"] else None, bn["name"], bn["address"]] if x))
        if f["remittance_info"]:
            out.append(f":70:{f['remittance_info']}")
        if s.get("missing") != "charges":
            ch = s["charges"] if s["charges"] in BAD_CHARGES else f["charges"]
            out.append(f":71A:{ch}")
        if r.random() < 0.3:  # partial paste: chopped header / trailing junk
            out.insert(0, r.choice(["pasted from SWIFT Alliance:", "msg below, pls check", "---"]))
        out.append("-}" if out[0].startswith("{1:") else r.choice(["", "-}", "...(rest cut off)"]))
        return "\n".join(out)

    def r_ticket(self, f, s):
        r = self.r
        oc, bn = f["ordering_customer"], f["beneficiary"]
        rows = [("Ticket", f"OPS-{r.randint(1000, 9999)}"), ("Ref", f["sender_reference"]),
                ("Ccy/Amt", f"{s['ccy']} | {s['amt']}")]
        if s.get("missing") != "value_date":
            rows.append(("VD", s["date"]))
        rows.append(("Ordering cust", self.party_s(oc, " ; ")))
        if f["ordering_institution"] or "badbic" in s and s["badbic"][0] == "ordering_institution":
            rows.append(("Ord inst (52A)", self.bic_s(f, s, "ordering_institution")))
        if s.get("missing") != "beneficiary":
            rows.append(("Bene", self.party_s(bn, " ; ")))
        if f["account_with_institution"] or "badbic" in s and s["badbic"][0] == "account_with_institution":
            rows.append(("AWI (57A)", self.bic_s(f, s, "account_with_institution")))
        if f["remittance_info"]:
            rows.append(("Details", f["remittance_info"]))
        if s.get("missing") != "charges":
            rows.append(("Chg", s["charges"]))
        if f["bank_operation_code"]:
            rows.append(("23B", f["bank_operation_code"]))
        sep = r.choice([": ", " = ", " - ", "\t"])
        return "\n".join(f"{k}{sep}{v}" for k, v in rows)

    def r_forward(self, f, s):
        head = "\n".join(["---------- Forwarded message ---------",
                          f"From: {self.person()} <ops{self.r.randint(1, 99)}@example.com>",
                          f"Subject: {self.r.choice(['RE: payment', 'FW: wire instructions', 'Urgent remittance', 'pymt req'])}", ""])
        inner = self.r_email(f, s) if self.r.random() < 0.5 else self.r_chat(f, s)
        return head + inner + "\n\n> earlier: pls ignore the draft I sent yesterday"


def split_ids(n: int, seed: int):
    ids = list(range(n))
    random.Random(seed + 1).shuffle(ids)
    a, b = int(n * 0.8), int(n * 0.9)
    return ids[:a], ids[a:b], ids[b:]


def generate(n: int, seed: int) -> list[dict]:
    g = Gen(seed)
    rows, seen = [], set()
    i = 0
    while len(rows) < n:
        ex = g.example(len(rows))
        i += 1
        h = hashlib.sha1(ex["text"].encode()).hexdigest()
        if h in seen:  # avoid duplicate texts across splits
            continue
        seen.add(h)
        rows.append(ex)
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=3000)
    ap.add_argument("--seed", type=int, default=103)
    ap.add_argument("--out", default=str(Path(__file__).parent))
    a = ap.parse_args()
    rows = generate(a.n, a.seed)
    tr, va, te = split_ids(len(rows), a.seed)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    counts = {}
    for name, idx in (("train", tr), ("val", va), ("test", te)):
        with open(out / f"{name}.jsonl", "w", encoding="utf-8") as fh:
            for i in sorted(idx):
                fh.write(json.dumps(rows[i], ensure_ascii=False) + "\n")
        counts[name] = len(idx)
    n_err = sum(1 for r in rows if r["target"]["errors"])
    manifest = {"synthetic": True, "note": "SYNTHETIC data generated by data/generate.py; not real payments.",
                "seed": a.seed, "n": len(rows), "splits": counts, "examples_with_errors": n_err}
    (out / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile mt103/evals/__init__.py

In [ ]:
%%writefile mt103/evals/run_eval.py
"""Shared evaluation for every model variant (base / fine-tuned / API reference / mock).

Examples:
  python evals/run_eval.py --backend ollama --model qwen2.5:0.5b-instruct --fewshot --name base-0.5b
  python evals/run_eval.py --backend ollama --model mt103-extractor --name ft-0.5b-smoke
  python evals/run_eval.py --backend claude-cli --model claude-haiku-5-5 --fewshot --name api-haiku
  python evals/run_eval.py --backend mock --limit 5 --name mock

Metrics: valid-JSON rate, schema-valid rate, field-level exact match (per field + macro),
error-flag precision/recall (micro over "CODE:field" keys, and example-level detection),
latency per example, output tokens/sec. Raw outputs are cached on disk
(results/cache/<name>.jsonl) so re-runs are free and resumable.
"""
from __future__ import annotations

import argparse
import concurrent.futures as cf
import datetime as dt
import hashlib
import json
import os
import re
import statistics
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from prompts import FEWSHOT_IDS, chat_messages, ft_messages, parse_output, target_str  # noqa: E402
from schema import Extraction, error_keys  # noqa: E402

SCORED_FIELDS = (
    "sender_reference", "bank_operation_code", "value_date", "currency", "amount",
    "ordering_customer.account", "ordering_customer.name", "ordering_customer.address",
    "ordering_institution", "account_with_institution",
    "beneficiary.account", "beneficiary.name", "beneficiary.address",
    "remittance_info", "charges",
)


# ---------------------------------------------------------------- scoring
def _get(fields, path):
    cur = fields
    for p in path.split("."):
        if not isinstance(cur, dict):
            return None
        cur = cur.get(p)
    return cur


def _norm(v):
    if v is None:
        return None
    if isinstance(v, str):
        s = re.sub(r"\s+", " ", v).strip().strip(".,;").casefold()
        return s or None
    return v


def field_match(path, gold, pred) -> bool:
    if path == "amount":
        if gold is None or pred is None:
            return gold is None and pred is None
        try:
            return abs(float(pred) - float(gold)) < 0.005
        except (TypeError, ValueError):
            return False
    if isinstance(pred, (int, float)) and not isinstance(pred, bool):
        pred = str(pred)
    return _norm(gold) == _norm(pred)


def score_one(gold: dict, pred_obj) -> dict:
    valid_json = isinstance(pred_obj, dict)
    schema_ok = False
    if valid_json:
        try:
            Extraction.model_validate(pred_obj)
            schema_ok = True
        except Exception:
            pass
    pf = pred_obj.get("fields") if valid_json and isinstance(pred_obj.get("fields"), dict) else {}
    per = {p: field_match(p, _get(gold["fields"], p), _get(pf, p)) for p in SCORED_FIELDS}
    gk = error_keys(gold["errors"])
    pk = error_keys(pred_obj.get("errors")) if valid_json else set()
    return {"valid_json": valid_json, "schema_valid": schema_ok, "fields": per,
            "err_tp": len(gk & pk), "err_fp": len(pk - gk), "err_fn": len(gk - pk),
            "has_err_gold": bool(gk), "has_err_pred": bool(pk)}


def aggregate(scores: list[dict], lat: list[float], toks: list[int], gen_s: list[float]) -> dict:
    n = len(scores)
    per = {p: sum(s["fields"][p] for s in scores) / n for p in SCORED_FIELDS}
    tp = sum(s["err_tp"] for s in scores)
    fp = sum(s["err_fp"] for s in scores)
    fn = sum(s["err_fn"] for s in scores)
    dtp = sum(s["has_err_gold"] and s["has_err_pred"] for s in scores)
    dfp = sum((not s["has_err_gold"]) and s["has_err_pred"] for s in scores)
    dfn = sum(s["has_err_gold"] and not s["has_err_pred"] for s in scores)
    div = lambda a, b: (a / b) if b else None  # noqa: E731
    return {
        "n": n,
        "valid_json_rate": sum(s["valid_json"] for s in scores) / n,
        "schema_valid_rate": sum(s["schema_valid"] for s in scores) / n,
        "field_accuracy_macro": sum(per.values()) / len(per),
        "field_accuracy": per,
        "all_fields_correct_rate": sum(all(s["fields"].values()) for s in scores) / n,
        "error_precision": div(tp, tp + fp), "error_recall": div(tp, tp + fn),
        "error_detect_precision": div(dtp, dtp + dfp), "error_detect_recall": div(dtp, dtp + dfn),
        "latency_mean_s": statistics.mean(lat) if lat else None,
        "latency_p50_s": statistics.median(lat) if lat else None,
        "latency_p95_s": (sorted(lat)[max(0, int(round(0.95 * len(lat))) - 1)] if lat else None),
        "tokens_per_sec": (sum(toks) / sum(gen_s)) if gen_s and sum(gen_s) > 0 else None,
    }


# ---------------------------------------------------------------- backends
def call_ollama(msgs, model, url, num_ctx):
    body = json.dumps({"model": model, "messages": msgs, "stream": False,
                       "options": {"temperature": 0, "num_predict": 600, "num_ctx": num_ctx, "num_thread": 1,
                                   "seed": 0}}).encode()
    req = urllib.request.Request(url.rstrip("/") + "/api/chat", data=body,
                                 headers={"Content-Type": "application/json"})
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=900) as r:
        out = json.loads(r.read())
    lat = time.time() - t0
    return {"raw": out["message"]["content"], "latency_s": lat,
            "out_tokens": out.get("eval_count", 0), "gen_s": out.get("eval_duration", 0) / 1e9,
            "prompt_tokens": out.get("prompt_eval_count", 0)}


def call_claude_cli(msgs, model):
    """Uses the logged-in `claude` CLI (no API key on this host). Few-shot is inlined."""
    sys_prompt = msgs[0]["content"]
    convo = []
    for m in msgs[1:-1]:
        tag = "INPUT" if m["role"] == "user" else "OUTPUT"
        convo.append(f"<{tag}>\n{m['content']}\n</{tag}>")
    prompt = ("Examples:\n" + "\n".join(convo) + "\n\n" if convo else "") + \
             f"<INPUT>\n{msgs[-1]['content']}\n</INPUT>\nReply with the OUTPUT JSON only."
    env = dict(os.environ, ENABLE_CLAUDEAI_MCP_SERVERS="false", CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC="1")
    cmd = ["claude", "-p", "--model", model, "--output-format", "json", "--tools", "",
           "--setting-sources", "", "--strict-mcp-config", "--no-session-persistence",
           "--system-prompt", sys_prompt, prompt]
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=300, env=env)
    lat = time.time() - t0
    if p.returncode != 0:
        raise RuntimeError(f"claude CLI failed: {p.stderr[-300:]}")
    out = json.loads(p.stdout)
    usage = out.get("usage") or {}
    api_ms = out.get("duration_api_ms") or 0
    return {"raw": out.get("result", ""), "latency_s": lat, "out_tokens": usage.get("output_tokens", 0),
            "gen_s": api_ms / 1000.0, "prompt_tokens": usage.get("input_tokens", 0)}


def call_mock(msgs, ex):
    """Deterministic mock for CI: returns the gold target, except it corrupts every 5th example."""
    raw = target_str(ex["target"])
    h = int(hashlib.md5(ex["id"].encode()).hexdigest(), 16)
    if h % 5 == 0:
        raw = "Sure! " + raw[: len(raw) // 2]  # truncated -> invalid JSON
    return {"raw": raw, "latency_s": 0.001, "out_tokens": len(raw) // 4, "gen_s": 0.001, "prompt_tokens": 0}


# ---------------------------------------------------------------- main
def load_jsonl(p):
    with open(p, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def run(args) -> dict:
    test = load_jsonl(args.data)[: args.limit] if args.limit else load_jsonl(args.data)
    fewshot = []
    if args.fewshot:
        train = {r["id"]: r for r in load_jsonl(Path(args.data).parent / "train.jsonl")}
        fewshot = [train[i] for i in FEWSHOT_IDS]
    cache_dir = Path(args.out_dir) / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_p = cache_dir / f"{args.name}.jsonl"
    cache = {}
    if cache_p.exists() and not args.no_cache:
        for r in load_jsonl(cache_p):
            cache[r["key"]] = r

    def key(ex, msgs):
        return hashlib.sha1(json.dumps([args.backend, args.model, msgs], ensure_ascii=False).encode()).hexdigest()

    def work(ex):
        msgs = ft_messages(ex["text"]) if args.prompt == "ft" else chat_messages(ex["text"], fewshot)
        k = key(ex, msgs)
        if k in cache:
            return ex, cache[k], True
        for attempt in range(3):
            try:
                if args.backend == "ollama":
                    res = call_ollama(msgs, args.model, args.ollama_url, args.num_ctx)
                elif args.backend == "claude-cli":
                    res = call_claude_cli(msgs, args.model)
                else:
                    res = call_mock(msgs, ex)
                break
            except Exception as e:  # noqa: BLE001
                if attempt == 2:
                    res = {"raw": None, "latency_s": None, "out_tokens": 0, "gen_s": 0, "error": str(e)[:300]}
                time.sleep(2 * (attempt + 1))
        res = {"key": k, "id": ex["id"], **res}
        return ex, res, False

    rows = []
    workers = args.workers if args.backend == "claude-cli" else 1
    with open(cache_p, "a", encoding="utf-8") as cfh, cf.ThreadPoolExecutor(workers) as pool:
        for i, (ex, res, hit) in enumerate(pool.map(work, test), 1):
            if not hit and args.backend != "mock" and res.get("raw") is not None:
                cfh.write(json.dumps(res, ensure_ascii=False) + "\n")
                cfh.flush()
            rows.append((ex, res))
            if i % 10 == 0 or i == len(test):
                print(f"[{args.name}] {i}/{len(test)}", flush=True)

    scores, lat, toks, gen = [], [], [], []
    per_example = []
    for ex, res in rows:
        obj = parse_output(res.get("raw"))
        s = score_one(ex["target"], obj)
        scores.append(s)
        if res.get("latency_s") is not None:
            lat.append(res["latency_s"])
        if res.get("gen_s"):
            toks.append(res.get("out_tokens", 0))
            gen.append(res["gen_s"])
        per_example.append({"id": ex["id"], "valid_json": s["valid_json"], "schema_valid": s["schema_valid"],
                            "fields_wrong": [k for k, v in s["fields"].items() if not v],
                            "err_fp": s["err_fp"], "err_fn": s["err_fn"]})
    summary = aggregate(scores, lat, toks, gen)
    summary.update({"name": args.name, "backend": args.backend, "model": args.model,
                    "fewshot": bool(args.fewshot), "prompt": args.prompt, "data": str(args.data),
                    "date": dt.date.today().isoformat(), "command": " ".join(sys.argv)})
    out = Path(args.out_dir) / f"eval_{args.name}.json"
    out.write_text(json.dumps({"summary": summary, "per_example": per_example}, indent=2, ensure_ascii=False))
    print(json.dumps({k: v for k, v in summary.items() if k != "field_accuracy"}, indent=2))
    return summary


def parse_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--backend", choices=["ollama", "claude-cli", "mock"], required=True)
    ap.add_argument("--model", default="")
    ap.add_argument("--name", required=True)
    ap.add_argument("--data", default=str(ROOT / "data" / "test.jsonl"))
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--fewshot", action="store_true")
    ap.add_argument("--prompt", choices=["full", "ft"], default="full",
                    help="full = long instruction prompt (+ few-shot); ft = short prompt the LoRA model was trained with")
    ap.add_argument("--ollama-url", default=os.environ.get("OLLAMA_URL", "http://127.0.0.1:21434"))
    ap.add_argument("--num-ctx", type=int, default=4096)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--out-dir", default=str(ROOT / "results"))
    ap.add_argument("--no-cache", action="store_true")
    return ap.parse_args(argv)


if __name__ == "__main__":
    run(parse_args())

In [ ]:
%%writefile mt103/train/config.yaml
# LoRA config. `mode` is chosen automatically (full if CUDA is available, else smoke)
# unless overridden with --mode.
base_model_cpu: Qwen/Qwen2.5-0.5B-Instruct
base_model_gpu: Qwen/Qwen2.5-1.5B-Instruct
lora:
  r: 16
  alpha: 32
  dropout: 0.05
  target_modules: [q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj]
train:
  lr: 2.0e-4
  scheduler: cosine
  warmup_ratio: 0.05
  max_length: 1024
  seed: 103
full:            # GPU: QLoRA 4-bit, 3 epochs over 2400 train examples
  epochs: 3
  batch_size: 8
  grad_accum: 2
  val_loss_examples: 300
  val_gen_examples: 50
  qlora: true
smoke:           # CPU: prove the pipeline end to end (NOT a fully trained model)
  max_steps: 150
  batch_size: 1
  grad_accum: 2          # effective batch 2 -> 300 training examples seen
  gradient_checkpointing: true   # keeps RAM under ~2.5 GB on the shared VPS
  eval_every: 50
  val_loss_examples: 40
  val_gen_examples: 20
  qlora: false

In [ ]:
%%writefile mt103/train/train_lora.py
"""LoRA / QLoRA fine-tuning of Qwen2.5-Instruct for MT103 extraction (TRL SFTTrainer).

    python train/train_lora.py                 # auto: full (GPU, QLoRA 4-bit) or smoke (CPU)
    python train/train_lora.py --mode smoke    # ~150 steps on CPU: pipeline proof, NOT fully trained

Outputs:
    models/adapter/                 LoRA adapter (+ tokenizer)
    results/train_log.json          loss per step, val loss per eval, val field accuracy
    results/loss.png                training / validation loss curve
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import yaml  # noqa: E402

from prompts import ft_messages, parse_output, target_str  # noqa: E402


def load_jsonl(p):
    with open(p, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def to_prompt_completion(rows):
    """Conversational prompt/completion format: TRL computes loss on the completion only."""
    return [{"prompt": ft_messages(r["text"]),
             "completion": [{"role": "assistant", "content": target_str(r["target"])}]} for r in rows]


def val_generation_accuracy(model, tok, rows, max_new_tokens=400):
    """Greedy-decode val examples and score them with the same scorer as evals/run_eval.py."""
    import torch
    from evals.run_eval import aggregate, score_one
    model.eval()
    scores, lat, toks, gen = [], [], [], []
    for r in rows:
        ids = tok.apply_chat_template(ft_messages(r["text"]), add_generation_prompt=True, return_tensors="pt",
                                      return_dict=True)
        ids = {k: v.to(model.device) for k, v in ids.items()}
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        dt_ = time.time() - t0
        new = out[0, ids["input_ids"].shape[1]:]
        raw = tok.decode(new, skip_special_tokens=True)
        scores.append(score_one(r["target"], parse_output(raw)))
        lat.append(dt_); toks.append(len(new)); gen.append(dt_)
    model.train()
    s = aggregate(scores, lat, toks, gen)
    return {k: s[k] for k in ("n", "valid_json_rate", "schema_valid_rate", "field_accuracy_macro",
                              "error_precision", "error_recall")}


def plot_loss(log, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
    tr = [(e["step"], e["loss"]) for e in log["history"] if "loss" in e]
    va = [(e["step"], e["eval_loss"]) for e in log["history"] if "eval_loss" in e]
    if tr:
        ax.plot(*zip(*tr), color="#2a6fdb", lw=1.6, label="train loss")
    if va:
        ax.plot(*zip(*va), color="#d9534f", lw=1.6, marker="o", label="val loss")
    ax.set_xlabel("step"); ax.set_ylabel("loss (completion tokens)")
    ax.set_title(f"LoRA on {log['base_model'].split('/')[-1]} — {log['mode']} run" +
                 (" (CPU smoke, not fully trained)" if log["mode"] == "smoke" else ""), fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.25); ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=str(ROOT / "train" / "config.yaml"))
    ap.add_argument("--mode", choices=["auto", "full", "smoke"], default="auto")
    ap.add_argument("--base-model", default=None, help="HF id or local path (overrides config)")
    ap.add_argument("--data-dir", default=str(ROOT / "data"))
    ap.add_argument("--out", default=str(ROOT / "models" / "adapter"))
    ap.add_argument("--results", default=str(ROOT / "results"))
    ap.add_argument("--max-steps", type=int, default=None)
    ap.add_argument("--val-gen-examples", type=int, default=None)
    a = ap.parse_args()

    import torch
    from datasets import Dataset
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
    from trl import SFTConfig, SFTTrainer

    cfg = yaml.safe_load(open(a.config))
    gpu = torch.cuda.is_available()
    mode = a.mode if a.mode != "auto" else ("full" if gpu else "smoke")
    mc = cfg[mode]
    if not gpu:
        torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "1")))
    base = a.base_model or (cfg["base_model_gpu"] if gpu else cfg["base_model_cpu"])
    local = ROOT / "models" / "base-qwen2.5-0.5b-instruct"
    if a.base_model is None and not gpu and local.exists():
        base = str(local)
    print(f"mode={mode} gpu={gpu} base={base}", flush=True)

    tok = AutoTokenizer.from_pretrained(base)
    kw = {}
    if gpu and mc.get("qlora"):
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16,
                                                       bnb_4bit_use_double_quant=True)
        kw["device_map"] = "auto"
    # CPU: bf16 weights (the VPS has avx512_bf16) to halve memory; LoRA weights stay fp32 (peft default).
    model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16 if gpu else torch.bfloat16, **kw)
    if gpu and mc.get("qlora"):
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model)

    train_rows = load_jsonl(Path(a.data_dir) / "train.jsonl")
    val_rows = load_jsonl(Path(a.data_dir) / "val.jsonl")
    train_ds = Dataset.from_list(to_prompt_completion(train_rows)).shuffle(seed=cfg["train"]["seed"])
    val_ds = Dataset.from_list(to_prompt_completion(val_rows[: mc["val_loss_examples"]]))
    gen_rows = val_rows[: (a.val_gen_examples if a.val_gen_examples is not None else mc["val_gen_examples"])]

    lc = cfg["lora"]
    peft_cfg = LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
                          target_modules=lc["target_modules"], task_type="CAUSAL_LM")
    tc = cfg["train"]
    steps_kw = {"num_train_epochs": mc["epochs"]} if mode == "full" else {"max_steps": a.max_steps or mc["max_steps"]}
    eval_kw = ({"eval_strategy": "epoch"} if mode == "full" else
               {"eval_strategy": "steps", "eval_steps": mc["eval_every"]})
    sft = SFTConfig(
        output_dir=str(ROOT / "train" / "runs" / mode), per_device_train_batch_size=mc["batch_size"],
        per_device_eval_batch_size=mc["batch_size"], gradient_accumulation_steps=mc["grad_accum"],
        learning_rate=tc["lr"], lr_scheduler_type=tc["scheduler"], warmup_steps=tc["warmup_ratio"],
        max_length=tc["max_length"], logging_steps=5, save_strategy="no", report_to=[],
        seed=tc["seed"], fp16=gpu, bf16=not gpu, use_cpu=not gpu, dataloader_num_workers=0,
        gradient_checkpointing=bool(mc.get("gradient_checkpointing", False)),
        **steps_kw, **eval_kw)

    log = {"mode": mode, "base_model": base, "gpu": gpu, "lora": lc, "train": tc, "mode_cfg": mc,
           "n_train": len(train_rows), "history": [], "val_generation": []}

    class GenEval(TrainerCallback):
        """Full mode: generation-based val accuracy after every epoch."""
        def on_epoch_end(self, args, state, control, model=None, **k):
            if mode == "full":
                m = val_generation_accuracy(model, tok, gen_rows)
                m.update({"epoch": state.epoch, "step": state.global_step})
                log["val_generation"].append(m)
                print("val_generation", m, flush=True)

    trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=val_ds,
                         processing_class=tok, peft_config=peft_cfg, callbacks=[GenEval()])
    t0 = time.time()
    trainer.train()
    log["train_seconds"] = round(time.time() - t0, 1)
    log["history"] = trainer.state.log_history
    if mode == "smoke":  # one generation-based val score at the end
        m = val_generation_accuracy(trainer.model, tok, gen_rows)
        m.update({"epoch": trainer.state.epoch, "step": trainer.state.global_step})
        log["val_generation"].append(m)
        print("val_generation", m, flush=True)

    out = Path(a.out)
    trainer.model.save_pretrained(out)
    tok.save_pretrained(out)
    (out / "TRAINING_INFO.json").write_text(json.dumps(
        {"mode": mode, "base_model": base if mode == "full" else cfg["base_model_cpu"],
         "steps": trainer.state.global_step, "epochs": trainer.state.epoch,
         "label": "CPU smoke run, not fully trained" if mode == "smoke" else "full QLoRA run"}, indent=2))
    res = Path(a.results); res.mkdir(exist_ok=True)
    (res / "train_log.json").write_text(json.dumps(log, indent=2, default=str))
    plot_loss(log, res / "loss.png")
    print(f"saved adapter to {out} in {log['train_seconds']}s", flush=True)


if __name__ == "__main__":
    main()

## Data (synthetic, regenerated + checked)

In [ ]:
%cd /kaggle/working/mt103
!python data/generate.py --n 3000 --seed 103 --out data
import hashlib
EXPECTED = {"train": "cedb68eb768c3cd83dc6e4f24a36262582340c9e787f4bf9c48d953cd17a4dd8", "val": "e498e9792a2b444e06d1914a6b05850f2d854050fd7441d88f4e58f39035bba5", "test": "3b488ecd75ae7c8d72ebd88a3db6bcd760965621486d9cc57ebb4a1e6715f845"}
DATA_DIR = "data"
for n, h in EXPECTED.items():
    got = hashlib.sha256(open(f"{DATA_DIR}/{n}.jsonl", "rb").read()).hexdigest()
    print(n, "OK" if got == h else f"MISMATCH (got {got[:12]}): splits differ from the repo's copy")

## Train (QLoRA, 3 epochs; val loss + val field accuracy logged every epoch)

In [ ]:
!python train/train_lora.py --mode full --data-dir $DATA_DIR --out models/adapter --results results

In [ ]:
import json
log = json.load(open("results/train_log.json"))
print("train seconds:", log["train_seconds"])
for v in log["val_generation"]:
    print(v)
from IPython.display import Image, display
display(Image("results/loss.png"))

## Package the adapter for download

In [ ]:
!cd models && zip -qr /kaggle/working/adapter.zip adapter && cd ..
!cp results/train_log.json results/loss.png /kaggle/working/
!ls -lh /kaggle/working/adapter.zip /kaggle/working/train_log.json /kaggle/working/loss.png
print("Download adapter.zip from the Output panel, unzip into models/adapter/ in the repo.")